# 01 — Click Interactions

The last lesson introduced the event system and showed how to extract coordinates from a click. This lesson uses those coordinates to **build things** — markers with labels, growing lists of points, a live path that draws itself as you click, and finally a GeoJSON export of everything you collected.

```
click  →  lat/lon  →  create feature  →  add to map
```

## Setup

In [ ]:
from ipyleaflet import Map, Marker, Polyline, GeoJSON
from ipywidgets import HTML

WICHITA_FALLS = (33.9137, -98.4934)

## Capturing Click Coordinates

Quick recap of the pattern from the last lesson — filter to `"click"` events, unpack `coordinates`.

```
click  →  kwargs["coordinates"]  →  (lat, lon)
```

Everything in this lesson starts from this two-line extract:

In [ ]:
def on_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    lat, lon = kwargs["coordinates"]   # always this line
    print(f"lat={lat:.5f}  lon={lon:.5f}")

m = Map(center=WICHITA_FALLS, zoom=12)
m.on_interaction(on_click)
m

## Numbered Markers with Popups

A bare marker is hard to tell apart once you have placed several. Add a sequential number and a popup that shows the exact coordinates.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)
count = [0]  # list so the closure can mutate it

def place_numbered_marker(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    count[0] += 1

    marker = Marker(location=(lat, lon), title=f"Point {count[0]}")
    marker.popup = HTML(value=f"""
        <b>Point {count[0]}</b><br>
        lat: {lat:.5f}<br>
        lon: {lon:.5f}
    """)
    m.add(marker)

m.on_interaction(place_numbered_marker)
m

Click several locations, then click each marker to see its popup.

## Storing Clicked Points

A marker is a visual object on the map — but it lives in ipyleaflet, not in Python. To use your clicked coordinates later (for computation, export, or building geometry), collect them into a plain Python list.

```python
clicked_points = []   # grows with every click
```

Each click appends one `(lat, lon)` tuple.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)
clicked_points = []

def collect_point(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    clicked_points.append((lat, lon))

    # Visual confirmation on the map
    m.add(Marker(location=(lat, lon), title=f"Point {len(clicked_points)}"))

    # Text feedback below the cell
    print(f"[{len(clicked_points)}]  ({lat:.5f}, {lon:.5f})")

m.on_interaction(collect_point)
m

In [ ]:
# Inspect the list after clicking
print(f"{len(clicked_points)} points collected:")
for i, (lat, lon) in enumerate(clicked_points):
    print(f"  {i+1}.  ({lat:.5f}, {lon:.5f})")

## Building a Live Path

Once coordinates are stored in a list, you can connect them into a line. A `Polyline` takes a list of `(lat, lon)` tuples and draws them as a connected path.

The trick: update the `Polyline`'s `locations` property each time a new point arrives. Because the polyline is a widget, the map redraws instantly — no cell re-run.

In [ ]:
m = Map(center=WICHITA_FALLS, zoom=12)

path_points = []
path_line = Polyline(locations=[], color="#e74c3c", weight=3)
m.add(path_line)

def build_path(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    path_points.append((lat, lon))

    # Drop a marker at the new point
    m.add(Marker(location=(lat, lon), title=f"Point {len(path_points)}"))

    # Update the line — widget redraws automatically
    path_line.locations = path_points.copy()

    print(f"Path has {len(path_points)} point(s)")

m.on_interaction(build_path)
m

Click several locations in sequence. Each click extends the red line to the new point.

The list of points in `path_points` is the data behind what you see — the map is just visualizing it.

## Exporting Clicked Points as GeoJSON

Collected coordinates are only useful if you can use them elsewhere. Convert `path_points` to a GeoJSON FeatureCollection and save it to a file.

Remember: GeoJSON coordinates are `[lon, lat]` — the reverse of ipyleaflet's `(lat, lon)`.

In [ ]:
import json

def points_to_geojson(points):
    """Convert a list of (lat, lon) tuples to a GeoJSON FeatureCollection."""
    features = [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]   # GeoJSON order: [lon, lat]
            },
            "properties": {"index": i + 1}
        }
        for i, (lat, lon) in enumerate(points)
    ]
    return {"type": "FeatureCollection", "features": features}

# Convert and inspect
fc = points_to_geojson(path_points)
print(json.dumps(fc, indent=2))

In [ ]:
# Save to file
output_path = "clicked_points.geojson"

with open(output_path, "w") as f:
    json.dump(fc, f, indent=2)

print(f"Saved {len(path_points)} points to {output_path}")

In [ ]:
# Reload the saved file and display it — closes the loop
with open(output_path) as f:
    saved = json.load(f)

m2 = Map(center=WICHITA_FALLS, zoom=12)
m2.add(GeoJSON(data=saved))
m2

## Exercise A

Click the `build_path` map above to collect at least 3 points. Then, in the cell below, convert `path_points` into a single **`LineString`** GeoJSON feature (not a FeatureCollection of Points). Print the result.

Remember: GeoJSON coordinate order is `[lon, lat]`.

In [ ]:
from ipyleaflet import GeoJSON

line_feature = {
    "type": "Feature",
    "properties": {"name": "Captured Path"},
    "geometry": {
        "type": "LineString",
        "coordinates": [[lon, lat] for lat, lon in path_points],
    },
}
print(line_feature)
GeoJSON(data={"type": "FeatureCollection", "features": [line_feature]})


## Exercise B

Add an **Undo** button to the live path builder. Each click of the button should:

1. Remove the last marker from the map
2. Pop the last entry from the points list
3. Update the `Polyline` to reflect the shorter path

In [ ]:
from ipywidgets import Button
from ipyleaflet import Polyline

undo_button = Button(description="Undo")

def redraw_path():
    global current_line
    if "current_line" in globals() and current_line in m.layers:
        m.remove_layer(current_line)
    if len(path_points) >= 2:
        current_line = Polyline(locations=path_points, color="blue", weight=3)
        m.add(current_line)

def undo_last(_):
    if markers:
        marker = markers.pop()
        m.remove_layer(marker)
    if path_points:
        path_points.pop()
    redraw_path()

undo_button.on_click(undo_last)
undo_button


```python
import json
from pathlib import Path
from ipyleaflet import Map, Marker

m = Map(center=(33.9137, -98.4934), zoom=11)
numbered_points = []


def handle_click(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs["coordinates"]
        number = len(numbered_points) + 1
        marker = Marker(location=(lat, lon), title=f"Point {number}")
        m.add(marker)
        numbered_points.append({"type": "Feature", "properties": {"index": number}, "geometry": {"type": "Point", "coordinates": [lon, lat]}})

m.on_interaction(handle_click)
Path("my_points.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": numbered_points}, indent=2))
m
```


## Next

In [02 — Dynamic Layers](./02-Dynamic_Layers.ipynb), we add and remove entire GeoJSON layers at runtime — toggling, swapping, and clearing them while the map is live.